In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import itertools
import joblib
from tqdm import tqdm

def analyze_and_predict_with_plot(pos_file, neg_file, val_file, max_k, output_excel, output_pred_xlsx):
    def load_data(file, label=None):
        with open(file, 'r') as f:
            seqs = [line.strip().upper() for line in f if line.strip()]
        df = pd.DataFrame({'seq': seqs})
        if label is not None:
            df['label'] = label
        return df

    df_pos = load_data(pos_file, 1)
    df_neg = load_data(neg_file, 0)
    df_val = load_data(val_file, label=-1)
    
    train_len = len(df_pos) + len(df_neg)
    df_all = pd.concat([df_pos, df_neg, df_val], ignore_index=True)
    
    seq_len = len(df_all['seq'][0])
    X_raw = np.array([list(s) for s in df_all['seq']])
    
    print(f"🧬 Total samples: {len(df_all)} (Train: {train_len}, To predict: {len(df_val)})")
    
    all_features = []
    feature_names = []
    
    for k_size in range(1, max_k + 1):
        print(f"Constructing {k_size}-base synergistic features...")
        comb_indices = list(itertools.combinations(range(seq_len), k_size))
        
        for idx_tuple in tqdm(comb_indices, desc=f"K={k_size}"):
            combined = X_raw[:, idx_tuple[0]].astype(object)
            for i in range(1, len(idx_tuple)):
                combined = combined + X_raw[:, idx_tuple[i]]
            
            all_features.append(combined)
            
            most_common = Counter(combined[:train_len]).most_common(1)[0][0]
            pos_str = "-".join([str(p+1) for p in idx_tuple])
            feature_names.append(f"K{k_size}_Pos{pos_str}({most_common})")
            
    X_encoded = np.zeros((len(df_all), len(all_features)), dtype=np.int32)
    le = LabelEncoder()
    for col in range(len(all_features)):
        X_encoded[:, col] = le.fit_transform(all_features[col])
    
    X_train_full = X_encoded[:train_len]
    y_train_full = df_all['label'][:train_len].values
    X_predict = X_encoded[train_len:]
    val_seqs = df_all['seq'][train_len:].values

    print("🚀 Training Random Forest model...")
    X_train, X_test, y_train, y_test = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    rf_model = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1, 
                                      max_features='sqrt', class_weight='balanced')
    rf_model.fit(X_train, y_train)
    
    test_acc = accuracy_score(y_test, rf_model.predict(X_test))
    print(f"✅ Test set accuracy: {test_acc:.2%}")
    
    rf_model.fit(X_train_full, y_train_full)

    preds = rf_model.predict(X_predict)
    probas = rf_model.predict_proba(X_predict)
    res_pred = pd.DataFrame({
        "Sequence": val_seqs,
        "Predicted Class": ["Positive" if p == 1 else "Negative" for p in preds],
        "Positive Probability": probas[:, 1].round(4),
        "Negative Probability": probas[:, 0].round(4)
    })
    res_pred.to_excel(output_pred_xlsx, index=False)

    results_imp = pd.DataFrame({'Feature': feature_names, 'Importance': rf_model.feature_importances_})
    results_imp = results_imp.sort_values(by='Importance', ascending=False).reset_index(drop=True)
    
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        results_imp.head(100).to_excel(writer, sheet_name='Top_100', index=False)
        results_imp.to_excel(writer, sheet_name='All_Ranking', index=False)

    print("📊 Plotting feature importance...")
    plt.figure(figsize=(12, 8), dpi=300)
    custom_color_list = [
        "#E282A7",
        "#E69EB6",
        "#F3C2D5",
        "#FBE0E9",
        "#FEEFF4",
        "#EDF2F6",
        "#DCE9F2",
        "#C2DBEF",
        "#97C2E4",
        "#4FA1D1"
    ]
    top_n = 10
    plot_data = results_imp.head(top_n)
    
    sns.barplot(data=plot_data, x='Importance', y='Feature', palette=custom_color_list)
    
    for i, v in enumerate(plot_data['Importance']):
        plt.text(v, i, f" {v:.4f}", va='center', fontsize=10, fontweight='bold')

    plt.title(f"Top {top_n} Synergistic Features (1-{max_k} bases)\nModel Accuracy: {test_acc:.2%}", fontsize=15)
    plt.xlabel("Gini Importance (Contribution to Decision)", fontsize=12)
    plt.ylabel("Synergistic Feature (Positions & Common Bases)", fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    plot_path = output_excel.replace('.xlsx', '.pdf')
    plt.savefig(plot_path)
    print(f"🖼️ Feature importance plot saved to: {plot_path}")
    plt.show()

    return res_pred

analyze_and_predict_with_plot(
    pos_file='./input/bind.txt',
    neg_file='./input/nobind.txt',
    val_file='./input/Mango-III_2.txt',
    max_k=3,
    output_excel='./output/Feature_Importance_Final.xlsx',
    output_pred_xlsx='./output/Predict_Results_Final.xlsx'
)
